# Local

In [43]:
import requests

In [ ]:
# Triton HTTP JSON API: tensores BYTES recebem strings cruas (sem base64)
payload = {
    "inputs": [
        {
            "name": "record_id",
            "shape": [1, 1],
            "datatype": "BYTES",
            "data": ["cardboard1"]
        },
        {
            "name": "image_s3_path",
            "shape": [1, 1],
            "datatype": "BYTES",
            "data": ["s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/cardboard1.jpg"]
        }
    ]
}

resp = requests.post("http://localhost:8080/v2/models/pipeline/infer", json=payload)
resp.raise_for_status()
resp.json()

{'model_name': 'pipeline', 'model_version': '1', 'parameters': {'sequence_id': 0, 'sequence_start': False, 'sequence_end': False}, 'outputs': [{'name': 'score', 'datatype': 'BYTES', 'shape': [1, 1], 'data': ['0.999529']}, {'name': 'class_name', 'datatype': 'BYTES', 'shape': [1, 1], 'data': ['cardboard']}, {'name': 'record_id', 'datatype': 'BYTES', 'shape': [1, 1], 'data': ['cardboard1']}]}

In [ ]:
# Extrair saídas: data[0] já é string decodificada
result = {o["name"]: o["data"][0] for o in resp.json()["outputs"]}
print(result)

{'score': '0.999529', 'class_name': 'cardboard', 'record_id': 'cardboard1'}


### Batch (múltiplos itens)

In [ ]:
payload = {
    "inputs": [
        {
            "name": "record_id",
            "shape": [2, 1],
            "datatype": "BYTES",
            "data": ["cardboard1", "metal1"]
        },
        {
            "name": "image_s3_path",
            "shape": [2, 1],
            "datatype": "BYTES",
            "data": [
                "s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/cardboard1.jpg",
                "s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/metal1.jpg"
            ]
        }
    ]
}

resp = requests.post("http://localhost:8080/v2/models/pipeline/infer", json=payload)
resp.raise_for_status()
resp.json()

{'model_name': 'pipeline', 'model_version': '1', 'parameters': {'sequence_id': 0, 'sequence_start': False, 'sequence_end': False}, 'outputs': [{'name': 'score', 'datatype': 'BYTES', 'shape': [2, 1], 'data': ['0.999529', '0.999757']}, {'name': 'class_name', 'datatype': 'BYTES', 'shape': [2, 1], 'data': ['cardboard', 'metal']}, {'name': 'record_id', 'datatype': 'BYTES', 'shape': [2, 1], 'data': ['cardboard1', 'metal1']}]}

In [ ]:
payload_batch = {
    "inputs": [
        {
            "name": "record_id",
            "shape": [5, 1],
            "datatype": "BYTES",
            "data": ["cardboard1", "metal1", "paper1", "plastic1", "trash1"]
        },
        {
            "name": "image_s3_path",
            "shape": [5, 1],
            "datatype": "BYTES",
            "data": [
                "s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/cardboard1.jpg",
                "s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/metal1.jpg",
                "s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/paper1.jpg",
                "s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/plastic1.jpg",
                "s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/trash1.jpg"
            ]
        }
    ]
}

resp_batch = requests.post("http://localhost:8080/v2/models/pipeline/infer", json=payload_batch)
resp_batch.raise_for_status()
resp_batch.json()

{'model_name': 'pipeline', 'model_version': '1', 'parameters': {'sequence_id': 0, 'sequence_start': False, 'sequence_end': False}, 'outputs': [{'name': 'score', 'datatype': 'BYTES', 'shape': [5, 1], 'data': ['0.999529', '0.999757', '0.998826', '0.999180', '0.998592']}, {'name': 'class_name', 'datatype': 'BYTES', 'shape': [5, 1], 'data': ['cardboard', 'metal', 'paper', 'plastic', 'trash']}, {'name': 'record_id', 'datatype': 'BYTES', 'shape': [5, 1], 'data': ['cardboard1', 'metal1', 'paper1', 'plastic1', 'trash1']}]}

# AWS

In [45]:
import boto3
import json
import time
import random
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse

In [46]:
session = boto3.session.Session()
region = session.region_name or "us-east-1"
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Região:     {region}")
print(f"Account ID: {account_id}")

Região:     us-east-1
Account ID: 891377318910


In [47]:
ECR_IMAGE_URI = f"{account_id}.dkr.ecr.{region}.amazonaws.com/sm-batch-transform-onnx:latest"
SAGEMAKER_ROLE_ARN = f"arn:aws:iam::{account_id}:role/SageMakerExecutionRole-DSA-P3"

S3_BUCKET = "mlops-us-east-1-891377318910"

S3_MODEL_KEY = "sagemaker-batch-transform/refinamento/model/model_onnx.tar.gz"
MODEL_DATA_URL = f"s3://{S3_BUCKET}/{S3_MODEL_KEY}"

INPUT_S3 = f"s3://{S3_BUCKET}/sagemaker-batch-transform/refinamento/input/"
OUTPUT_S3 = f"s3://{S3_BUCKET}/sagemaker-batch-transform/refinamento/output/"

print(f"Imagem ECR:      {ECR_IMAGE_URI}")
print(f"Role ARN:        {SAGEMAKER_ROLE_ARN}")
print(f"Model data URL:  {MODEL_DATA_URL}")
print(f"Input S3:        {INPUT_S3}")
print(f"Output S3:       {OUTPUT_S3}")

Imagem ECR:      891377318910.dkr.ecr.us-east-1.amazonaws.com/sm-batch-transform-onnx:latest
Role ARN:        arn:aws:iam::891377318910:role/SageMakerExecutionRole-DSA-P3
Model data URL:  s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/model/model_onnx.tar.gz
Input S3:        s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/
Output S3:       s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/output/


### Criar Input Metadata File

In [48]:
_parsed = urlparse(INPUT_S3)
_bucket = _parsed.netloc
_prefix = _parsed.path.lstrip("/")

_pag = boto3.client("s3").get_paginator("list_objects_v2")
all_image_keys = [
    obj["Key"]
    for page in _pag.paginate(Bucket=_bucket, Prefix=_prefix)
    for obj in page.get("Contents", [])
    if obj["Key"].lower().endswith((".jpg", ".jpeg", ".png"))
]

print(f"Imagens disponíveis no prefixo: {len(all_image_keys)}")

Imagens disponíveis no prefixo: 45


In [49]:
# Cada linha é um request Triton completo (KServe v2) — formato exigido pelo Batch Transform
_parsed = urlparse(INPUT_S3)
_bucket = _parsed.netloc

n_records = 45
_sample = random.sample(all_image_keys, n_records)

_lines = [
    json.dumps({
        "inputs": [
            {
                "name": "record_id",
                "shape": [1, 1],
                "datatype": "BYTES",
                "data": [f"r{i:04d}"]
            },
            {
                "name": "image_s3_path",
                "shape": [1, 1],
                "datatype": "BYTES",
                "data": [f"s3://{_bucket}/{key}"]
            }
        ]
    })
    for i, key in enumerate(_sample)
]

metadata_key = "sagemaker-batch-transform/refinamento/metadata/metadata_onnx.jsonl"
boto3.client("s3").put_object(
    Bucket=_bucket,
    Key=metadata_key,
    Body="\n".join(_lines).encode("utf-8"),
    ContentType="application/jsonlines",
)

metadata_s3_uri = f"s3://{_bucket}/{metadata_key}"
print(f"Metadata S3 URI: {metadata_s3_uri}")
print(f"\nExemplo da 1ª linha:\n{_lines[0]}")

Metadata S3 URI: s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/metadata/metadata_onnx.jsonl

Exemplo da 1ª linha:
{"inputs": [{"name": "record_id", "shape": [1, 1], "datatype": "BYTES", "data": ["r0000"]}, {"name": "image_s3_path", "shape": [1, 1], "datatype": "BYTES", "data": ["s3://mlops-us-east-1-891377318910/sagemaker-batch-transform/refinamento/input/cardboard8.jpg"]}]}


### Criar SageMaker Model

In [50]:
sm_client = boto3.client("sagemaker", region_name=region)

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d%H%M")
model_name = f"resnet18-classifier-onnx-{timestamp}"
print(f"Nome do model: {model_name}")

Nome do model: resnet18-classifier-onnx-20260621154936


In [ ]:
response_model = sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": ECR_IMAGE_URI,
        "ModelDataUrl": MODEL_DATA_URL,
        # SAGEMAKER_TRITON_DEFAULT_MODEL_NAME: obrigatório para ensemble (doc AWS)
        "Environment": {
            "SAGEMAKER_TRITON_DEFAULT_MODEL_NAME": "pipeline",
            "ALLOWED_S3_BUCKETS": S3_BUCKET,
            "SAGEMAKER_TRITON_SHM_DEFAULT_BYTE_SIZE": "67108864",  # 64 MB para o Python backend
        },
    },
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN,
)

model_arn = response_model["ModelArn"]
print(f"Model ARN: {model_arn}")

Model ARN: arn:aws:sagemaker:us-east-1:891377318910:model/resnet18-classifier-onnx-20260621154936


### Criar Batch Transform Job

In [ ]:

job_name = f"batch-job-onnx-{timestamp}"
instance_type = "ml.g4dn.xlarge"
instance_count = 1
max_payload = 6

response_job = sm_client.create_transform_job(
    TransformJobName=job_name,
    ModelName=model_name,
    MaxConcurrentTransforms=1,
    # MultiRecord: SageMaker agrupa linhas até MaxPayloadInMB num único POST /invocations.
    # O server.py divide em chunks de até 8 (max_batch_size do classifier) e
    # devolve N linhas JSON individuais no mesmo formato KServe v2.
    BatchStrategy="MultiRecord",
    MaxPayloadInMB=max_payload,
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": metadata_s3_uri,
            }
        },
        "ContentType": "application/json",
        "SplitType": "Line",
    },
    TransformOutput={
        "S3OutputPath": OUTPUT_S3,
        "AssembleWith": "Line",
    },
    TransformResources={
        "InstanceType": instance_type,
        "InstanceCount": instance_count,
    },
)

job_arn = response_job["TransformJobArn"]

print(f"Nome do job: {job_name}")
print(f"Job ARN: {job_arn}")


### Monitorar Status do Job

In [ ]:
def describe_job(name):
    desc = sm_client.describe_transform_job(TransformJobName=name)
    return {
        "status": desc["TransformJobStatus"],
        "start": desc.get("TransformStartTime"),
        "end": desc.get("TransformEndTime"),
        "failure": desc.get("FailureReason", ""),
    }

# Polling até o job terminar
while True:
    info = describe_job(job_name)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Status: {info['status']}")
    if info["status"] in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(30)

print(f"\nFinalizado: {info['status']}")
if info["failure"]:
    print(f"Falha: {info['failure']}")
else:
    print(f"Início: {info['start']}")
    print(f"Fim:    {info['end']}")

### Ler Resultados do S3

In [ ]:
_parsed_out = urlparse(OUTPUT_S3)
_out_bucket = _parsed_out.netloc
_out_prefix = _parsed_out.path.lstrip("/")

_s3 = boto3.client("s3")
_pag = _s3.get_paginator("list_objects_v2")

output_keys = [
    obj["Key"]
    for page in _pag.paginate(Bucket=_out_bucket, Prefix=_out_prefix)
    for obj in page.get("Contents", [])
    if obj["Key"].endswith(".jsonl") or obj["Key"].endswith(".out")
]

print(f"Arquivos de saída encontrados: {len(output_keys)}")
for k in output_keys:
    print(f"  s3://{_out_bucket}/{k}")

In [ ]:
records = []
for key in output_keys:
    body = _s3.get_object(Bucket=_out_bucket, Key=key)["Body"].read().decode()
    for line in body.strip().splitlines():
        if not line:
            continue
        resp_json = json.loads(line)
        out = {o["name"]: o["data"][0] for o in resp_json.get("outputs", [])}
        records.append(out)

df = pd.DataFrame(records)[["record_id", "class_name", "score"]]
df["score"] = df["score"].astype(float)
df = df.sort_values("record_id").reset_index(drop=True)

print(f"Total de registros: {len(df)}")
df